In [1]:
# --- 1. Import Libraries ---
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# --- 2. Load Dataset ---
df = pd.read_csv("cleaned_croma_laptops.csv")

# --- 3. Features & Target ---
X = df.drop("Price", axis=1)   # predictors
y = df["Price"]                # target (price)

# --- 4. Train-Test Split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- 5. Identify Categorical & Numeric Features ---
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_features = X.select_dtypes(include=[np.number]).columns

# --- 6. Preprocessing ---
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# --- 7. Build Random Forest Model ---
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,   # number of trees
        max_depth=None,     # let trees grow fully
        random_state=42,
        n_jobs=-1           # use all CPU cores
    ))
])

# --- 8. Train Model ---
rf_model.fit(X_train, y_train)

# --- 9. Predictions ---
y_pred = rf_model.predict(X_test)

# --- 10. Evaluation Metrics ---
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Random Forest Regression Results")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)


Random Forest Regression Results
MAE : 16127.35529033393
RMSE: 27565.31715486558
R²  : 0.8217827761155698


In [2]:
import pickle

# Save the trained Ridge model
with open("random_forest_model.pkl", "wb") as file:
    pickle.dump(rf_model, file)